# Détection de Flux Réseau - Modèle Compatible

Ce notebook crée un modèle compatible avec le code des correcteurs (labels texte).

**Meilleur modèle : Random Forest avec 98.84%**

In [1]:
# Imports
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.metrics import classification_report
import skops.io as skio
import warnings
warnings.filterwarnings('ignore')

print("✓ Imports réussis")

✓ Imports réussis


In [2]:
# Classe de preprocessing personnalisée
class NetworkFlowPreprocessor(BaseEstimator, TransformerMixin):
    """Preprocessing automatique pour les flux réseau"""
    
    def __init__(self):
        self.feature_names_ = None
        self.medians_ = {}
    
    def fit(self, X, y=None):
        # Nettoyer
        X_clean = X.replace([np.inf, -np.inf], np.nan)
        
        # Calculer médianes
        for col in X_clean.columns:
            if X_clean[col].isnull().sum() > 0:
                self.medians_[col] = X_clean[col].median()
        
        # Remplir
        X_filled = X_clean.copy()
        for col, median_val in self.medians_.items():
            X_filled[col].fillna(median_val, inplace=True)
        
        # Garder colonnes non-constantes
        mask = X_filled.std() > 0
        self.feature_names_ = X_filled.columns[mask].tolist()
        
        return self
    
    def transform(self, X):
        X_clean = X.replace([np.inf, -np.inf], np.nan)
        X_filled = X_clean.copy()
        
        # Remplir avec médianes du fit
        for col in X_filled.columns:
            if col in self.medians_:
                X_filled[col].fillna(self.medians_[col], inplace=True)
            elif X_filled[col].isnull().sum() > 0:
                X_filled[col].fillna(0, inplace=True)
        
        # Filtrer colonnes
        cols_present = [c for c in self.feature_names_ if c in X_filled.columns]
        return X_filled[cols_present]

print("✓ Classe de preprocessing créée")

✓ Classe de preprocessing créée


In [3]:
# Chargement des données
df = pd.read_parquet("Training.parquet")
print(f"Dataset chargé: {df.shape[0]:,} lignes, {df.shape[1]} colonnes")

X = df.drop(columns=["ClassLabel"])
y = df["ClassLabel"]

print(f"\nClasses: {sorted(y.unique())}")
print(f"Distribution:\n{y.value_counts()}")

Dataset chargé: 6,417,302 lignes, 58 colonnes

Classes: ['Benign', 'Botnet', 'Bruteforce', 'DDoS', 'DoS', 'Infiltration', 'Portscan', 'Webattack']
Distribution:
ClassLabel
Benign          5030332
DDoS             864310
DoS              278140
Botnet           102177
Bruteforce        72270
Infiltration      66399
Webattack          2096
Portscan           1578
Name: count, dtype: int64


In [4]:
# Split pour validation
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Train: {X_train.shape[0]:,} | Validation: {X_val.shape[0]:,}")

Train: 5,133,841 | Validation: 1,283,461


In [5]:
# Créer le pipeline avec Random Forest (supporte labels texte)
print("Création du pipeline Random Forest...")

pipeline = Pipeline([
    ('preprocessor', NetworkFlowPreprocessor()),
    ('scaler', StandardScaler()),
    ('classifier', RandomForestClassifier(
        n_estimators=200,
        max_depth=15,
        min_samples_split=5,
        min_samples_leaf=2,
        random_state=42,
        n_jobs=-1
    ))
])

print("✓ Pipeline créé")

Création du pipeline Random Forest...
✓ Pipeline créé


In [6]:
# Entraînement sur validation
import time

print("Entraînement sur le train set...")
start = time.time()

pipeline.fit(X_train, y_train)

elapsed = time.time() - start
print(f"✓ Entraînement terminé en {elapsed/60:.1f} minutes")

# Scores
train_score = pipeline.score(X_train, y_train)
val_score = pipeline.score(X_val, y_val)

print(f"\nScore Train: {train_score:.4f} ({train_score*100:.2f}%)")
print(f"Score Validation: {val_score:.4f} ({val_score*100:.2f}%)")

Entraînement sur le train set...
✓ Entraînement terminé en 12.5 minutes

Score Train: 0.9881 (98.81%)
Score Validation: 0.9880 (98.80%)


In [7]:
# Rapport de classification
y_pred = pipeline.predict(X_val)
print("\nClassification Report:")
print(classification_report(y_val, y_pred))


Classification Report:
              precision    recall  f1-score   support

      Benign       0.99      1.00      0.99   1006067
      Botnet       1.00      0.98      0.99     20435
  Bruteforce       1.00      0.99      1.00     14454
        DDoS       1.00      1.00      1.00    172862
         DoS       0.99      0.99      0.99     55628
Infiltration       0.80      0.03      0.05     13280
    Portscan       0.98      0.75      0.85       316
   Webattack       1.00      0.64      0.78       419

    accuracy                           0.99   1283461
   macro avg       0.97      0.80      0.83   1283461
weighted avg       0.99      0.99      0.98   1283461



In [8]:
# Entraînement sur TOUTES les données
print("=" * 60)
print("ENTRAÎNEMENT FINAL SUR TOUTES LES DONNÉES")
print("=" * 60)

start = time.time()
pipeline.fit(X, y)
elapsed = time.time() - start

final_score = pipeline.score(X, y)

print(f"\n✓ Entraînement terminé en {elapsed/60:.1f} minutes")
print(f"Score final: {final_score:.4f} ({final_score*100:.2f}%)")

ENTRAÎNEMENT FINAL SUR TOUTES LES DONNÉES

✓ Entraînement terminé en 16.0 minutes
Score final: 0.9882 (98.82%)


In [9]:
# Sauvegarde
print("\nSauvegarde du modèle...")

skio.dump(pipeline, "student_model.skio")

import os
size_mb = os.path.getsize("student_model.skio") / (1024 * 1024)
print(f"✓ Modèle sauvegardé: student_model.skio ({size_mb:.2f} MB)")


Sauvegarde du modèle...
✓ Modèle sauvegardé: student_model.skio (116.83 MB)


In [10]:
# VALIDATION - Code des correcteurs
print("=" * 60)
print("VALIDATION AVEC LE CODE DES CORRECTEURS")
print("=" * 60)

# Charger le modèle
unknown_types = skio.get_untrusted_types(file="student_model.skio")
print(f"Types: {unknown_types}")

with open("student_model.skio", "rb") as f:
    loaded_model = skio.load(f, trusted=unknown_types)

print("✓ Modèle chargé")

# Tester avec données BRUTES (comme les correcteurs)
df_test = pd.read_parquet("Training.parquet")
X_test = df_test.drop(columns=["ClassLabel"])
y_test = df_test["ClassLabel"]

# Score avec labels TEXTE (pas encodés!)
test_score = loaded_model.score(X_test, y_test)

print(f"\n{'=' * 60}")
print(f"SCORE FINAL: {test_score:.4f} ({test_score*100:.2f}%)")
print(f"{'=' * 60}")
print("\n✓ Modèle compatible avec le code des correcteurs!")
print("✓ Utilise les labels texte directement")
print("✓ Preprocessing intégré dans le pipeline")

VALIDATION AVEC LE CODE DES CORRECTEURS
Types: ['__main__.NetworkFlowPreprocessor']
✓ Modèle chargé

SCORE FINAL: 0.9882 (98.82%)

✓ Modèle compatible avec le code des correcteurs!
✓ Utilise les labels texte directement
✓ Preprocessing intégré dans le pipeline


Code lionel test (a peine modifier)

In [16]:
import pandas as pd
df = pd.read_parquet("Training.parquet")
import skops.io as skio
from_model = None
with open("student_model.skio", "rb") as f:
    loaded_model = skio.load(f, trusted=unknown_types)
X = df.drop(columns=["ClassLabel"], axis=1)
y = df["ClassLabel"]
test_score = loaded_model.score(X_test, y_test)

print(test_score)

0.9881727554663938
